# Diagnostics · Drainage-density threshold calibration

**Primary interface** for the Earth drainage-density calibration. Calls
`channel_heads.dd_calibration` only — the same functions the batch wrapper
`scripts/diagnostics/calibrate_stream_threshold_by_mars_dd.py` uses:
`collect_basin_metrics_for_dem`, `summarize_threshold_metrics`,
`choose_best_threshold` (+ the `MARS_DD_STATS` Mars target).

It is **read-only**: it sweeps thresholds for **one** basin in memory and
displays the result. It writes no CSVs / plots. (The batch wrapper sweeps all
17 basins.)

> Runs TopoToolbox stream extraction once per threshold for one DEM — takes a
> minute or two.

In [1]:
from dataclasses import asdict

import pandas as pd

from channel_heads.basin_config import LOCAL_TO_PAPER_BASIN, get_basin_config
from channel_heads.config import EXAMPLE_DEMS
from channel_heads.dd_calibration import (
    MARS_DD_STATS,
    choose_best_threshold,
    collect_basin_metrics_for_dem,
    summarize_threshold_metrics,
)

# Reduced threshold set for a snappy single-basin demo (the batch run uses the
# full DEFAULT_THRESHOLDS_KM2 across all basins).
THRESHOLDS_KM2 = [1, 5, 20, 50]

basin = next((b for b in EXAMPLE_DEMS if EXAMPLE_DEMS[b].exists()), None)
print("basin:", basin, "->", EXAMPLE_DEMS[basin] if basin else "none on disk")

basin: inyo -> /Users/guypi/Projects/channel-heads/data/cropped_DEMs/Inyo_strm_crop.tif


## Sweep thresholds for one basin — via the package

In [2]:
if basin is not None:
    cfg = get_basin_config(LOCAL_TO_PAPER_BASIN.get(basin, basin))
    rows = collect_basin_metrics_for_dem(
        dem_path=EXAMPLE_DEMS[basin],
        basin_name=basin,
        thresholds_km2=THRESHOLDS_KM2,
        z_th=cfg["z_th"],
        lat_deg=float(cfg["lat"]),
    )
    per_basin = pd.DataFrame([asdict(r) for r in rows])
    print(f"{len(per_basin)} (outlet-basin, threshold) rows for {basin}")
else:
    per_basin = pd.DataFrame()
    print("no DEM on disk — skipping")

66 (outlet-basin, threshold) rows for inyo


## Drainage density vs threshold + chosen threshold

In [3]:
if not per_basin.empty:
    summary = summarize_threshold_metrics(per_basin, mars_stats=MARS_DD_STATS)
    best = choose_best_threshold(summary)
    print(f"Mars Dd target: {MARS_DD_STATS}")
    print(f"Chosen best threshold (score_iqr): {best} km^2")
    cols = [c for c in ["threshold_km2", "median_dd", "mean_dd", "score_iqr"]
            if c in summary.columns]
    display(summary[cols].round(4) if cols else summary.round(4))
else:
    print("nothing to summarise")

Mars Dd target: {'n': 391, 'median': 0.2238836162, 'mean': 0.241869091, 'q1': 0.1611662654, 'q3': 0.2986920715, 'iqr': 0.1375258061, 'std': 0.1189935099}
Chosen best threshold (score_iqr): 5.0 km^2


,threshold_km2,score_iqr
0,1.0,19.1854
1,5.0,13.2019
2,20.0,106.1686


---
Full calibration (all basins; writes CSVs, GeoPackage and plots):

```bash
python scripts/diagnostics/calibrate_stream_threshold_by_mars_dd.py --mars-target -v
```